# Rilevamento vertici faccia cubo — training YOLOv8n-pose

Step 3 del piano "vision": alleniamo un rilevatore dei 4 vertici per ogni faccia visibile del cubo, sul dataset sintetico generato da `web/vision/dataset/generate-dataset.ts` (una classe, 4 keypoint, formato Ultralytics pose).

**Prima di eseguire**: `Runtime > Cambia tipo di runtime > GPU (T4 va bene)`.

Questo notebook non fa altro che allenare + esportare: la generazione del dataset e il controllo di correttezza dei keypoint sono gia' stati fatti e verificati a parte (Step 1-2).

In [ ]:
!nvidia-smi

## 1. Dataset — scegli UNA delle due opzioni

**Opzione A (consigliata, piu' veloce)**: il dataset e' gia' stato generato in locale (6000 train + 800 val, ~180MB). Carica lo zip su Google Drive una volta, poi scaricalo qui.

**Opzione B**: rigeneralo direttamente in Colab (nessun upload, ma reinstalla Node/Chrome ogni sessione — utile solo se vuoi piu' immagini o una variazione diversa senza dover ricaricare lo zip).

### Opzione A — dataset gia' pronto, da Google Drive

1. Carica `web/vision/dataset/cube-face-keypoints-dataset.zip` (generato in locale) in `MyDrive/rubik-vision/` sul tuo Google Drive.
2. Esegui la cella sotto (chiede il permesso di accesso a Drive).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ZIP_PATH = '/content/drive/MyDrive/rubik-vision/cube-face-keypoints-dataset.zip'  #@param {type:"string"}
DATASET_DIR = '/content/dataset'

assert os.path.exists(ZIP_PATH), f"Non trovo {ZIP_PATH}: hai caricato lo zip su Drive nel percorso giusto?"
!mkdir -p {DATASET_DIR}
!unzip -q -o "{ZIP_PATH}" -d {DATASET_DIR}
!ls {DATASET_DIR}

DATA_YAML = f'{DATASET_DIR}/data.yaml'
print('DATA_YAML =', DATA_YAML)

### Opzione B — rigenera in Colab (salta se hai usato l'opzione A)

Richiede che il branch sia gia' pushato su GitHub: `git push -u origin feat/vision-face-keypoints` dal tuo repo locale, PRIMA di eseguire queste celle.

In [ ]:
%%bash
set -e
NODE_VERSION=22.13.0
curl -fsSL https://nodejs.org/dist/v${NODE_VERSION}/node-v${NODE_VERSION}-linux-x64.tar.xz -o /tmp/node.tar.xz
mkdir -p /opt/node
tar -xJf /tmp/node.tar.xz -C /opt/node --strip-components=1
ln -sf /opt/node/bin/node /usr/local/bin/node
ln -sf /opt/node/bin/npm /usr/local/bin/npm
ln -sf /opt/node/bin/npx /usr/local/bin/npx
node --version

In [ ]:
REPO_URL = 'https://github.com/AndreaAzzarello/rubik-solve-coach.git'  #@param {type:"string"}
BRANCH = 'feat/vision-face-keypoints'  #@param {type:"string"}
TRAIN_COUNT = 6000  #@param {type:"integer"}
VAL_COUNT = 800  #@param {type:"integer"}

!git clone --branch {BRANCH} --single-branch {REPO_URL} /content/repo
!cd /content/repo/web && npm install -g pnpm && pnpm install --frozen-lockfile
!cd /content/repo/web && npx playwright install-deps chromium
!cd /content/repo/web && pnpm bench:setup
!cd /content/repo/web && node --experimental-strip-types vision/dataset/generate-dataset.ts {TRAIN_COUNT} {VAL_COUNT}

DATA_YAML = '/content/repo/web/vision/dataset/output/data.yaml'
print('DATA_YAML =', DATA_YAML)

## 2. Training

`DATA_YAML` deve gia' esistere (dalla cella dell'opzione A o B sopra).

In [ ]:
MODEL_VARIANT = 'yolov8n-pose.pt'  #@param ["yolov8n-pose.pt", "yolov8s-pose.pt", "yolo11n-pose.pt"]
EPOCHS = 100  #@param {type:"integer"}
IMG_SIZE = 512  #@param {type:"integer"}

assert 'DATA_YAML' in dir(), "Esegui prima la cella dell'opzione A o B per definire DATA_YAML"

!pip install -q ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_VARIANT)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=-1,  # batch size automatico in base alla memoria GPU disponibile
    project='runs',
    name='cube_face_keypoints',
)

In [ ]:
# Metriche Ultralytics native (mAP/OKS pose) sul val set. La metrica specifica
# del piano (PCK / grid-cell hit-rate, vedi conversazione) e' il prossimo step,
# non ancora incluso qui.
metrics = model.val()
print(metrics)

## 3. Export ONNX

In [ ]:
onnx_path = model.export(format='onnx', opset=12, simplify=True, imgsz=IMG_SIZE)
print(onnx_path)

## 4. Salva i risultati (scegli una delle due celle)

In [ ]:
# Opzione 1: copia su Google Drive (persiste tra sessioni)
from google.colab import drive
drive.mount('/content/drive')
import shutil, os
DEST = '/content/drive/MyDrive/rubik-vision/models'
os.makedirs(DEST, exist_ok=True)
shutil.copy(onnx_path, DEST)
shutil.copy(str(model.trainer.best), DEST)
print(f'Copiati in {DEST}')

In [ ]:
# Opzione 2: download diretto nel browser
from google.colab import files
files.download(onnx_path)

## Prossimo passo

Il `.onnx` esportato va copiato in `web/vision/models/cube-face-keypoints.onnx` nel repo. Prima di integrarlo nel browser, il piano prevede una verifica di correttezza sul modello stesso: metrica PCK e grid-cell hit-rate sul val set sintetico (Step successivo, non ancora fatto), poi un controllo di trasferimento su qualche frame reale prima di toccare `lib/video-decoder.ts`.